# Notebook 07 — Solutions to Every Exercise

> **Easiest way to run this: Google Colab — nothing to install.**
> Go to https://colab.research.google.com then File > Upload notebook and choose this file.
> Prefer your own computer? Lesson 1 shows the VS Code and local-Jupyter paths.

One cell per exercise. Try your own version first, then run the matching cell to check it.
The comment on each line shows what you should see.

In [ ]:
# Run this once. Sets up everything the solutions need.
%pip install -q numpy sentence-transformers chromadb
import numpy as np
import math

# The three similarity helpers from Lesson 5, used by several solutions below.
def dot_product(a, b):
    return sum(x * y for x, y in zip(a, b))

def magnitude(v):
    return math.sqrt(sum(x * x for x in v))

def cosine_similarity(a, b):
    return dot_product(a, b) / (magnitude(a) * magnitude(b))

print("Ready.")

## Lesson 2 — Make Your Own Vector

In [ ]:
# Build the vector (5, 2) and read its numbers.
v = np.array([5, 2])

print("v =", v)
print("v[0] =", v[0])    # 5
print("v[1] =", v[1])    # 2

## Lesson 3 — Make the Sum Come Out Right (choose b)

In [ ]:
# Choose b so that a + b prints [10 10]: work the addition backwards.
a = np.array([2, 5])
b = np.array([8, 5])          # (10-2, 10-5)

print("a + b =", a + b)       # [10 10]
print("3 * a =", 3 * a)       # [6 15]

## Lesson 3 — Fix the Broken Cell (the lists trap)

In [ ]:
# The bug: plain lists, where + JOINS instead of adding.
a = [4, 1]
b = [1, 3]
print(a + b)              # [4, 1, 1, 3]  <- concatenated, not added!

# The fix: NumPy arrays, where + adds matching numbers.
a = np.array([4, 1])
b = np.array([1, 3])
print(a + b)              # [5 4]

## Lesson 4 — Lengths Forwards and Backwards

In [ ]:
# Forwards: the length of (5, 12).
print(np.linalg.norm(np.array([5, 12])))   # 13.0

In [ ]:
# Backwards: DIFFERENT vectors whose length is exactly 5 (any of these counts).
print(np.linalg.norm(np.array([4, 3])))    # 5.0
print(np.linalg.norm(np.array([5, 0])))    # 5.0
print(np.linalg.norm(np.array([0, 5])))    # 5.0

In [ ]:
# Backwards: choose x so (x, 0) has length 7 — a straight arrow's length IS its number.
print(np.linalg.norm(np.array([7, 0])))    # 7.0

## Lesson 4 — Distance Between Two Points

In [ ]:
# Distance from (1, 1) to (4, 5).
p = np.array([1, 1])
q = np.array([4, 5])
print("distance:", np.linalg.norm(q - p))        # 5.0

## Lesson 5 — Dot Products by Hand

In [ ]:
# Two dot products, using the helper defined in the first cell.
print("(2,3) . (1,4) =", dot_product([2, 3], [1, 4]))   # 14
print("(1,0) . (0,5) =", dot_product([1, 0], [0, 5]))   # 0  (right angle)

## Lesson 5 — Cosine by Hand

In [ ]:
# Cosine similarity of two vectors that point the same way.
a = [1, 2]
b = [2, 4]
print("cosine (1,2),(2,4):", round(cosine_similarity(a, b), 3))   # 1.0

## Lesson 5 — Write Your Own Function

In [ ]:
# Define a tiny function and call it.
def triple(x):
    return 3 * x

print("triple(4) =", triple(4))   # 12

## Lesson 6 — Compare Two People (5 numbers each)

In [ ]:
# Predict who Kim is closer to, then check.
alex = [0.90, 0.30, 0.80, 0.20, 0.50]
jo   = [0.60, 0.70, 0.10, 0.90, 0.80]
kim  = [0.62, 0.68, 0.12, 0.88, 0.78]

print("Kim vs Alex:", round(cosine_similarity(kim, alex), 3))   # 0.703
print("Kim vs Jo  :", round(cosine_similarity(kim, jo), 3))     # 1.0  -> closer to Jo

## Lesson 8 — Embed Your Own Sentences
This and the later cells use the real model and the vector store (installed in the first cell).

In [ ]:
# Embed three sentences and compare a similar pair with an unrelated one.
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("all-MiniLM-L6-v2")

texts = ["I enjoy hiking in the mountains",
         "Walking up hills is my favourite",   # similar to the first
         "The printer is out of ink"]          # unrelated
v = model.encode(texts)

print("similar pair  :", round(cosine_similarity(v[0], v[1]), 3))   # higher
print("unrelated pair:", round(cosine_similarity(v[0], v[2]), 3))   # lower

## Lesson 9 — Ask Your Store Something
First build the store...

In [ ]:
# Build a small store of sentences.
import chromadb
from chromadb.utils import embedding_functions

ef = embedding_functions.SentenceTransformerEmbeddingFunction(model_name="all-MiniLM-L6-v2")
client = chromadb.Client()

sentences = [
    "The dog wagged its tail when its owner came home.",
    "A kitten chased a ball of yarn across the floor.",
    "Lions live in prides on the African savanna.",
    "I drove the car to the grocery store this morning.",
    "The pizza was hot and covered in melted mozzarella.",
]

try:
    client.delete_collection("sentences")
except Exception:
    pass
col = client.create_collection(name="sentences", embedding_function=ef,
                               metadata={"hnsw:space": "cosine"})
col.add(ids=[f"s{i}" for i in range(len(sentences))], documents=sentences)
print("stored", col.count(), "sentences")

...then STEER a query: make the bicycle sentence the top hit (without saying
bicycle, electric, or miles). Plus the bracket-bug fix.

In [ ]:
# Steer the search: aim at the bicycle sentence by meaning alone.
for q in ["two wheels you pedal around town",
          "a battery-powered ride with two wheels"]:
    result = col.query(query_texts=[q], n_results=1)
    print(q, "->", result["documents"][0][0])

In [ ]:
# The bracket bug: [0] reaches the result LIST; the second [0] reaches the sentence.
result = col.query(query_texts=["a fast car"], n_results=1)
print(result["documents"][0])      # ['...']  <- a list, with brackets
print(result["documents"][0][0])   # the sentence itself

## Lesson 10 — Search Your Own Library
Build a tiny movie library...

In [ ]:
# Build a store of movie descriptions.
movies = [
    "A young wizard discovers he is famous in a hidden magical world.",
    "Dinosaurs are brought back to life on a remote island theme park.",
    "A team of astronauts travels through a wormhole to save humanity.",
    "Toys come to life whenever people leave the room.",
    "A hobbit carries a powerful ring across a dangerous land to destroy it.",
]

try:
    client.delete_collection("movies")
except Exception:
    pass
lib = client.create_collection(name="movies", embedding_function=ef,
                               metadata={"hnsw:space": "cosine"})
lib.add(ids=[f"m{i}" for i in range(len(movies))], documents=movies)
print("indexed", lib.count(), "movies")

...then search it with words none of the descriptions use.

In [ ]:
# Search with a phrase that shares no words with any description.
result = lib.query(query_texts=["a story about space travel"], n_results=2)

for doc, dist in zip(result["documents"][0], result["distances"][0]):
    print(f"similarity {1 - dist:.3f}   {doc}")     # the astronauts / wormhole movie wins

## Lesson 11 — Your FAQ Finder
Build the FAQ store (questions stored, answers attached as metadata)...

In [ ]:
# Build the FAQ store.
faq = [
    ("What are your opening hours?", "We're open 11am to 11pm every day, including weekends."),
    ("Do you offer vegan options?", "Yes - we have plant-based cheese and several vegetable-only pizzas."),
    ("Do you deliver?", "We deliver free within 5 km; a small fee applies beyond that."),
    ("Is there parking nearby?", "There's a free public car park right behind the restaurant."),
]
questions = [q for q, a in faq]
answers = [a for q, a in faq]

try:
    client.delete_collection("faq")
except Exception:
    pass
fcol = client.create_collection(name="faq", embedding_function=ef,
                                metadata={"hnsw:space": "cosine"})
fcol.add(ids=[f"f{i}" for i in range(len(faq))],
         documents=questions,
         metadatas=[{"answer": a} for a in answers])
print("indexed", fcol.count(), "FAQ questions")

...then write `ask()` and try it on reworded questions.

In [ ]:
# Match a reworded question and return the stored answer.
def ask(question):
    r = fcol.query(query_texts=[question], n_results=1)
    print("Q:", question)
    print("A:", r["metadatas"][0][0]["answer"], "\n")

ask("Can you bring it to my house?")     # -> the delivery answer
ask("Where do I leave the car?")         # -> the parking answer

## Lesson 8 — Watch Context Steer the Word "Bank" 

In [ ]:
# Both sentences contain "bank"; the probes reveal which meaning each carries.
sentences = [
    "We sat on the river bank and watched the water.",      # A
    "I deposited my salary at the bank this morning.",       # B
    "a river with flowing water",                             # probe 1
    "money and finance",                                      # probe 2
]
v = model.encode(sentences)

print("A vs water probe:", round(cosine_similarity(v[0], v[2]), 3))   # 0.531 high
print("B vs water probe:", round(cosine_similarity(v[1], v[2]), 3))   # 0.098
print("A vs money probe:", round(cosine_similarity(v[0], v[3]), 3))   # 0.190
print("B vs money probe:", round(cosine_similarity(v[1], v[3]), 3))   # 0.301 higher

## Lesson 8 — Find the Cut-Off (truncation)

In [ ]:
# Text past 256 tokens is silently ignored.
filler = "The committee reviewed the quarterly schedule and noted the agenda. " * 40

probe = model.encode("pizza and italian food")
at_end   = model.encode(filler + " The secret topic of this document is pizza.")
at_front = model.encode("The secret topic of this document is pizza. " + filler)

print("pizza at the END  :", round(cosine_similarity(probe, at_end), 3))    # ~0.102
print("pizza at the FRONT:", round(cosine_similarity(probe, at_front), 3))  # ~0.231

## Lesson 12 — Tokenizer exercises

In [ ]:
# Take the tokenizer for a spin + count tokens vs words.
for text in ["extraordinary", "QX-99turbo"]:
    pieces = model.tokenizer.tokenize(text)
    print(f"{text:<15} {pieces}   ({len(pieces)} tokens)")

s = "The quick brown fox jumps over the lazy dog."
print("tokens in 9-word sentence:", len(model.tokenizer.tokenize(s)))   # 10

In [ ]:
# The expensive-phrase hunt: 3 "words", 10 tokens.
s = "COVID-19 don't k8s"
print(len(model.tokenizer.tokenize(s)))    # 10
print(model.tokenizer.tokenize(s))

In [ ]:
# The storage bill: 1M documents at 384 dims.
docs = 1_000_000
print(round(docs * 384 * 4 / 1024 / 1024), "MB")   # ~1465 MB ~ 1.5 GB

## Lesson 13 — Gotcha exercises (negation, numbers, pooling label)

In [ ]:
# Negation: opposite policies, near-identical vectors.
A = model.encode("Returns are accepted within 30 days.")
B = model.encode("Returns are not accepted within 30 days.")
q = model.encode("are returns accepted?")
print("query vs A:", round(cosine_similarity(q, A), 3))   # 0.776
print("query vs B:", round(cosine_similarity(q, B), 3))   # 0.715
print("A vs B    :", round(cosine_similarity(A, B), 3))   # 0.915

In [ ]:
# Numbers: 2 days vs 20 days.
a = model.encode("Your order ships within 2 days.")
b = model.encode("Your order ships within 20 days.")
print(round(cosine_similarity(a, b), 3))    # 0.904

In [ ]:
# Read the model's own label: pooling + max length.
print(model)   # Pooling: pooling_mode_mean_tokens True; Transformer: max_seq_length 256

## Lesson 2 — Build the Vector I'm Thinking Of

In [ ]:
# v[0] must show 2 and v[1] must show 9 -> the vector is (2, 9).
v = np.array([2, 9])
print(v[0])   # 2
print(v[1])   # 9

## Lesson 5 — Fix the Broken Ruler

In [ ]:
# The bug: magnitude without the square root.
def bad_magnitude(v):
    return sum(x * x for x in v)              # missing sqrt!

print(dot_product([2, 3], [2, 3]) / (bad_magnitude([2, 3]) * bad_magnitude([2, 3])))
# 0.0769...  <- a vector vs ITSELF should be 1.0; the ruler is broken

# The fix (this is the correct magnitude defined in the first cell):
print(cosine_similarity([2, 3], [2, 3]))      # 1.0

## Lesson 8 — Try to Break the 384

In [ ]:
# Every input gives 384 numbers; the dimension belongs to the model, not the text.
print(len(model.encode("a")))                                  # 384
print(len(model.encode("an extremely long sentence " * 20)))   # 384
print(len(model.encode("zorptab90")))                          # 384

## Lesson 10 — Predict the Top Two (space travel)

In [ ]:
# Against the 20-document library from notebook 05
# (rebuilt here so this solutions notebook runs on its own).
documents = [
    # Space
    "Astronauts aboard the space station experience weightlessness for months.",
    "A telescope gathers faint light from distant galaxies.",
    "The rover collected rock samples from the surface of Mars.",
    "Comets grow a glowing tail as they near the sun.",
    # Cooking
    "Let the bread dough rise for an hour before baking.",
    "Simmer the tomatoes slowly to deepen the sauce's flavour.",
    "Whisk the eggs until the mixture turns pale and fluffy.",
    "Season the soup with a pinch of salt and fresh herbs.",
    # Money
    "Paying off high-interest debt first saves you the most money.",
    "A diversified portfolio spreads risk across many assets.",
    "Compound interest grows your savings faster over time.",
    "Set aside an emergency fund before you start investing.",
    # Health
    "Regular walking lowers blood pressure and lifts your mood.",
    "Drinking enough water keeps you alert through the afternoon.",
    "Stretching before exercise reduces the chance of injury.",
    "A good night's sleep helps your body repair itself.",
    # Technology
    "The new laptop boots in seconds thanks to its fast drive.",
    "Encryption keeps your messages private from eavesdroppers.",
    "Cloud storage lets you reach your files from any device.",
    "A strong password is long, unusual, and hard to guess.",
]

try:
    client.delete_collection("library")
except Exception:
    pass
lib20 = client.create_collection(name="library", embedding_function=ef,
                                 metadata={"hnsw:space": "cosine"})
lib20.add(ids=[f"doc_{i}" for i in range(len(documents))], documents=documents)

def search(query, k=3):
    result = lib20.query(query_texts=[query], n_results=k)
    for doc, dist in zip(result["documents"][0], result["distances"][0]):
        print(f"  similarity {1 - dist:.3f}   {doc}")

search("space travel", k=2)
#   the space-station sentence, then the telescope sentence (the rover ranks last!)

## Lesson 11 — Produce Exactly ['q5', 'q6', 'q7']

In [ ]:
# range with a start AND a stop.
print([f"q{i}" for i in range(5, 8)])    # ['q5', 'q6', 'q7']

## Lesson 13 — Read the Label, Imagine It Different

In [ ]:
# Mean pooling + 256 tokens, straight off the printout.
print(model)
# If it used CLS pooling, the Pooling line would show
# 'pooling_mode_cls_token': True and 'pooling_mode_mean_tokens': False.

That covers every exercise. The full FAQ build (all ten pairs) is in
`06_faq_finder.ipynb`.